In [1]:
import os
import cv2
import mediapipe as mp
from tqdm import tqdm


In [2]:
Og_data = "../data/UTKFace"
crop_data = "../data/cropped"

os.makedirs(crop_data, exist_ok=True)
Og_data, crop_data


('../data/UTKFace',
 '../data/cropped')

In [3]:
mp_face = mp.solutions.face_detection
detector = mp_face.FaceDetection(model_selection=0, min_detection_confidence=0.5)


In [4]:
files = os.listdir(Og_data)
len(files)


24109

In [5]:
def get_age_from_filename(filename):
    return int(filename.split("_")[0])


In [7]:
def crop_face(image, detection, pad=0.25):
    bbox = detection.location_data.relative_bounding_box
    
    h, w, _ = image.shape

    x = int(bbox.xmin * w)
    y = int(bbox.ymin * h)
    w_box = int(bbox.width * w)
    h_box = int(bbox.height * h)
    
    x -= int(w_box * pad)   #Earlier testing showed a padding of 0.25 was pretty good for getting the important features of face for training
    y -= int(h_box * pad)
    w_box += int(2 * w_box * pad)
    h_box += int(2 * h_box * pad)

    x = max(0, x) #There is a chance the data might overflow out of the image so set the boundaries 
    y = max(0, y)
    w_box = min(w_box, w - x)
    h_box = min(h_box, h - y)

    face = image[y:y+h_box, x:x+w_box]
    face = cv2.resize(face, (224, 224))
    
    return face


In [12]:
saved = 0
skipped = 0

for file in tqdm(files):
    path = os.path.join(Og_data, file)
    img = cv2.imread(path)
    if img is None:
        skipped += 1
        continue
    results = detector.process(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    if not results.detections:
        skipped += 1
        continue

    try:
        face = crop_face(img, results.detections[0])
    except:
        skipped += 1
        continue

    
    age = get_age_from_filename(file)
    age_dir = os.path.join(crop_data, str(age))
    os.makedirs(age_dir, exist_ok=True)

    save_path = os.path.join(age_dir, file)
    if not save_path.lower().endswith((".jpg", ".jpeg", ".png")):# Having some issues with the file type so had to ensure that the file tyoe is the one wanted
        save_path += ".jpg"
    if face is not None and face.size != 0:
        cv2.imwrite(save_path, face)
        saved += 1
    else:
        skipped += 1
saved, skipped #getting the saved and skipped images


100%|████████████████████████████████████████████████████████████████████████████| 24109/24109 [08:43<00:00, 46.06it/s]


(23987, 122)

In [2]:
import torch
torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None

(False, None)